# 49 - Encode the scaled corpus with BGE-M3

Same structure as notebook 45/48, `BAAI/bge-m3` instead, same settings as `16_baseline_bgem3.ipynb` (`normalize_embeddings=True`, no instruction prefix needed).

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

RESULT_DIR = Path("result/49_encode_bgem3_scaled")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RESULT_DIR / "company_embeddings_checkpoint.npy"
FINAL_PATH = RESULT_DIR / "company_embeddings.npy"
CHUNK_SIZE = 20_000

combined = pd.read_parquet("result/44_build_scaled_corpus/combined_pool.parquet")
rich_texts = combined["rich_text"].tolist()
print(f"[Load] Companies to encode: {len(rich_texts):,}", flush=True)

print(f"[GPU] CUDA available : {torch.cuda.is_available()}", flush=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"[GPU] Device : {torch.cuda.get_device_name(0)}", flush=True)

# Reverted: jobs 6089651/6089883 looked hung (near-zero CPU, no progress lines in the log) but
# were NOT -- CPU stays near-zero during GPU-bound encoding regardless, and the chunk files on
# disk (320,000 rows, verified non-corrupted) prove real progress happened during both jobs.
# The progress print()s just never reached the .out file because stdout was buffered and the
# 30-min TIME_LIMIT kill didn't flush it before exiting. This is the same "needs several
# resubmissions" pattern as every other slow encoder here, not a loading hang -- reverted to
# the original SentenceTransformer approach so the already-encoded chunks stay consistent with
# whatever encodes the rest. flush=True added to every print below so future logs show real
# progress instead of looking empty.
print("[Encode] Loading BGE-M3 (BAAI/bge-m3)...", flush=True)
t0 = time.time()
os.environ["HF_HUB_OFFLINE"] = "1"
try:
    model = SentenceTransformer("BAAI/bge-m3", device=DEVICE, local_files_only=True)
    print("[Encode] Loaded from local cache -- skipped Hugging Face Hub network calls", flush=True)
except Exception as e:
    print(f"[Encode] Not fully cached locally yet ({type(e).__name__}) -- retrying with network access (this will be slower)", flush=True)
    os.environ.pop("HF_HUB_OFFLINE", None)
    model = SentenceTransformer("BAAI/bge-m3", device=DEVICE)
print(f"[Encode] Model loaded in {time.time()-t0:.1f}s on {model.device}", flush=True)
batch_size = 128
print(f"[Encode] Batch size : {batch_size}", flush=True)

In [ ]:
CHUNKS_DIR = RESULT_DIR / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)


def atomic_save_npy(arr, path):
    """Write to a temp file then atomically rename -- a plain np.save() left a truncated,
    corrupted checkpoint on notebook 51 when a job was killed mid-write. Same fix applied here."""
    p = Path(path)
    tmp_path = p.with_suffix(".tmp" + p.suffix)
    np.save(tmp_path, arr)
    os.replace(tmp_path, path)


def get_chunk_files():
    return sorted(CHUNKS_DIR.glob("chunk_*.npy"), key=lambda p: int(p.stem.split("_")[1]))


if FINAL_PATH.exists() and np.load(FINAL_PATH, mmap_mode="r").shape[0] == len(rich_texts):
    print("[Encode] Final embeddings already on disk -- skipping", flush=True)
    embeddings = np.load(FINAL_PATH)
else:
    chunk_files = get_chunk_files()
    start = sum(np.load(f, mmap_mode="r").shape[0] for f in chunk_files)

    # One-time migration: an older run may have left a single ever-growing checkpoint file
    # (the pattern that triggered a bwUniCluster high-I/O warning -- 746GB written against only
    # 43GB of actual data, since it rewrote the full accumulated array on every chunk). Fold
    # whatever it already has into the new per-chunk format once, instead of re-encoding it.
    if CHECKPOINT_PATH.exists() and start == 0:
        legacy = np.load(CHECKPOINT_PATH)
        print(f"[Encode] Migrating legacy checkpoint ({legacy.shape[0]:,} rows) into per-chunk format...", flush=True)
        atomic_save_npy(legacy, CHUNKS_DIR / f"chunk_{0:09d}.npy")
        start = legacy.shape[0]
        CHECKPOINT_PATH.unlink()
        chunk_files = get_chunk_files()

    if start:
        print(f"[Encode] Resuming -- {start:,}/{len(rich_texts):,} already encoded across {len(chunk_files)} chunk files", flush=True)

    t0 = time.time()
    for chunk_start in range(start, len(rich_texts), CHUNK_SIZE):
        chunk_texts = rich_texts[chunk_start:chunk_start + CHUNK_SIZE]
        chunk_embs = model.encode(
            chunk_texts,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        # Save ONLY this chunk as its own small file -- never rewrite everything already saved.
        atomic_save_npy(chunk_embs, CHUNKS_DIR / f"chunk_{chunk_start:09d}.npy")
        done_so_far = chunk_start + len(chunk_texts)
        elapsed = time.time() - t0
        print(f"[Encode] {done_so_far:,}/{len(rich_texts):,} encoded ({elapsed/60:.1f} min elapsed)", flush=True)

    # Assemble the final array exactly once, only now that every chunk is done.
    chunk_files = get_chunk_files()
    embeddings = np.concatenate([np.load(f) for f in chunk_files], axis=0)
    atomic_save_npy(embeddings, FINAL_PATH)
    for f in chunk_files:
        f.unlink()
    print(f"[Encode] Done. Embeddings shape: {embeddings.shape}", flush=True)
    print(f"[Encode] Saved -> {FINAL_PATH}", flush=True)